# Matched sweeps on Colab

The notebook uses the corrected implementation for five seeds, 42 to 46.
Each seed changes folds and model initialization. Results are separate from
historical reports and may differ after the scoring and BLIP corrections.

1. Prepare the intended local inputs and create their manifest with
   `python scripts/input_manifest.py --write`. Review the chosen population.
2. If you choose to use Colab, place the private `data/` tree under
   `MyDrive/bmpb/inputs/`. Keep its `processed/input_manifest.json` with it.
3. Select a GPU runtime, then run the cells in order. Check the recorded source
   commit and input verification before starting training.

The code revision is pinned. Model revisions and software versions are recorded
by the runner. Completed runs save to `MyDrive/bmpb/review_seeds/` and resume
only when their identities and artifacts pass verification. Runtime and cost
have not been measured for the corrected implementation.


In [ ]:
# Cell 1: Check the selected GPU
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'{gpu}, {mem:.0f} GB')


In [ ]:
# Cell 2: Mount Drive for saved results
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/bmpb/review_seeds'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f'Results will save to {DRIVE_OUT}')


In [ ]:
# Cell 3: Load the selected source and record its commit
import json, os, subprocess, sys
from pathlib import Path

REPO = '/content/bangla-multimodal-political-stance'
SOURCE_REPO = os.environ.get(
    'BMPB_SOURCE_REPO',
    'https://github.com/WhiteHades/bangla-multimodal-political-stance.git',
)
SOURCE_REF = os.environ.get(
    'BMPB_SOURCE_REF', '8ae7c74a50f51bf5191dfcec2b2ffc01b81150b8',
)
if not Path(REPO).exists():
    subprocess.run(['git', 'clone', '--no-checkout', '--depth', '1', SOURCE_REPO, REPO], check=True)
else:
    remote = subprocess.check_output(
        ['git', '-C', REPO, 'remote', 'get-url', 'origin'], text=True,
    ).strip()
    if remote != SOURCE_REPO:
        raise RuntimeError('The existing checkout has a different source repository.')
    if subprocess.check_output(['git', '-C', REPO, 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('The existing checkout has local changes. Preserve them before using a clean checkout.')
subprocess.run(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', SOURCE_REF], check=True)
subprocess.run(['git', '-C', REPO, 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(REPO)
SOURCE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
SOURCE_RECORD = Path('/content/bmpb-source-checkout.json')
SOURCE_RECORD.write_text(json.dumps({
    'repository': SOURCE_REPO, 'source_ref': SOURCE_REF, 'commit': SOURCE_COMMIT,
}, indent=2))
print(f'Source {SOURCE_REF} resolved to {SOURCE_COMMIT}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[fetch]'], check=True)


In [ ]:
# Cell 4: Restore and verify the chosen inputs before training
import os, shutil, subprocess, sys
from pathlib import Path

os.chdir(REPO)
INPUT_ROOT = Path(os.environ.get('BMPB_INPUT_ROOT', '/content/drive/MyDrive/bmpb/inputs'))
source_data = INPUT_ROOT / 'data'
if not (source_data / 'processed/input_manifest.json').is_file():
    raise RuntimeError('Supply the chosen data tree and its frozen input manifest before training.')
shutil.copytree(source_data, Path(REPO) / 'data', dirs_exist_ok=True)
subprocess.run([sys.executable, 'scripts/input_manifest.py'], check=True)

import pandas as pd
df = pd.read_csv(Path(REPO) / 'data/processed/corpus.csv')
print(f'Verified corpus: {len(df)} items')
print(f'Labels: {df["label_name"].value_counts().to_dict()}')
print('The runner will check actual image decoding when selecting the paired population.')


In [ ]:
# Cell 5: Run the matched sweep for all five seeds
#
# Each seed saves to Drive incrementally. If the runtime disconnects,
# rerun this cell. Completed runs are verified before reuse.

import os, subprocess, sys, json, shutil
from pathlib import Path

os.chdir('/content/bangla-multimodal-political-stance')

SEEDS = [42, 43, 44, 45, 46]
DRIVE_OUT = '/content/drive/MyDrive/bmpb/review_seeds'

env = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'TOKENIZERS_PARALLELISM': 'false',
}

for seed in SEEDS:
    out_dir = f'{DRIVE_OUT}/seed-{seed}'
    summary = Path(out_dir) / 'sweep_summary.json'

    # The runner validates each run identity and required artifact before skipping it.
    if summary.exists():
        print(f'Seed {seed}: checking saved run identities')
    else:
        print(f'Seed {seed}: starting fresh')

    print(f'\n{"="*60}')
    print(f'=== SEED {seed} ===')
    print(f'{"="*60}', flush=True)

    result = subprocess.run(
        [sys.executable, '-u', 'scripts/run_matched_sweep.py',
         '--seed', str(seed), '--out', out_dir],
        env=env,
    )

    shutil.copy(SOURCE_RECORD, Path(out_dir) / 'source-checkout.json')

    if result.returncode != 0:
        print(f'!!! Seed {seed} had errors (check logs above)')
    else:
        print(f'--- Seed {seed} complete ---')

# Summary
print(f'\n{"="*60}')
print('=== ALL SEEDS SUMMARY ===')
for seed in SEEDS:
    summary = Path(f'{DRIVE_OUT}/seed-{seed}/sweep_summary.json')
    if summary.exists():
        rows = json.loads(summary.read_text())
        n_ok = sum(1 for r in rows if r.get('status') == 'ok')
        n_fail = sum(1 for r in rows if r.get('status') == 'failed')
        print(f'  Seed {seed}: {n_ok} ok, {n_fail} failed')
    else:
        print(f'  Seed {seed}: not started')

In [ ]:
# Cell 6: Aggregate verified seeds and download the results
import os, subprocess, sys, json, tarfile
from pathlib import Path

os.chdir(REPO)
subprocess.run([
    sys.executable, '-u', 'scripts/seed_analysis.py',
    *[f'{DRIVE_OUT}/seed-{seed}' for seed in SEEDS],
    '--dest', f'{DRIVE_OUT}/analysis',
], check=True)

print('Per seed macro F1 on the paired population')
for seed in SEEDS:
    summary = Path(f'{DRIVE_OUT}/seed-{seed}/sweep_summary.json')
    rows = json.loads(summary.read_text())
    paired = [r for r in rows if r.get('population') == 'paired_headlines' and r.get('status') == 'ok']
    for r in sorted(paired, key=lambda x: x.get('macro_f1', 0), reverse=True):
        print(f"seed={seed} {r['name']:20s} F1={r['macro_f1']:.3f}")

archive_path = '/content/bmpb-review-seeds.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(DRIVE_OUT, arcname='review_seeds')
print(f'Archive: {archive_path} ({Path(archive_path).stat().st_size / 1e6:.1f} MB)')

from google.colab import files
files.download(archive_path)
